# HW1c - a liver biopsy, filtered

**Where the hand-built toolkit meets real data.** Chapter 6 (CNNs), attached to
**L16 CNN Foundations**. No PyTorch, no GPU, no training.

> **Notebook 2 of 2.** **HW1b** built the tools: a convolution engine, a filter menu, and
> HSV. This one points them at a real liver biopsy and asks them to measure something a
> pathologist actually cares about.
>
> This is the **task** notebook. Solutions: `HW1c_biopsy_solution.ipynb`.

It works. Then you move to a second slide and it stops working - and that is the argument
for everything after L16.

### What you will do

| Task | | |
|---|---|---|
| 1 | 🧀🧀 | blur as denoising, and what it costs you |
| 2 | 🧀🧀 | why one edge kernel is never enough |
| 3 | 🧀🧀🧀 | separate the two stains - RGB fails, HSV works |
| 4 | 🧀🧀🧀 | search the thresholds, and notice what is missing |
| 5 | 🧀🧀🧀 | count the nuclei, then decide what the count is worth |
| 6 | 🧀🧀🧀 | quantify fibrosis and steatosis on a second stain |
| 7 | 🧀 | the honesty question |

### Setup

The helpers below are the ones you wrote in HW1b. They are repeated here so this notebook
stands alone - read them, do not re-derive them.

In [ ]:
import time
from pathlib import Path
from urllib.request import Request, urlopen

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import hsv_to_rgb, rgb_to_hsv
from PIL import Image
from scipy import ndimage          # ONE library call, in task 5. Everything else is yours.

SEED = 509
rng = np.random.default_rng(SEED)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["image.cmap"] = "gray"

RAW = ("https://raw.githubusercontent.com/HaykTarkhanyan/"
       "python_math_ml_course/main/ml/ch6_cnn/")


def load_image(relpath):
    """Local checkout first; on Colab download once and cache. Raises if both fail."""
    local = Path(relpath)
    if not local.exists():
        local.parent.mkdir(parents=True, exist_ok=True)
        url = RAW + relpath.replace("\\", "/")
        req = Request(url, headers={"User-Agent": "python-math-ml-course/hw1c"})
        with urlopen(req, timeout=60) as r:
            local.write_bytes(r.read())
        print(f"downloaded {url} -> {local}")
    return np.asarray(Image.open(local).convert("RGB"), dtype=float) / 255.0


def show(images, titles, ncols=None, size=3.0, suptitle=None):
    n = len(images); ncols = ncols or n
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(size * ncols, size * 1.14 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, im, t in zip(axes, images, titles):
        ax.imshow(np.clip(im, 0, 1)); ax.set_title(t, fontsize=10); ax.axis("off")
    for ax in axes[n:]:
        ax.axis("off")
    if suptitle:
        fig.suptitle(suptitle, fontsize=12, y=1.01)
    fig.tight_layout(); plt.show()


def to_gray(rgb):
    return rgb @ np.array([0.299, 0.587, 0.114])


def convolve2d_fast(img, kernel, pad_mode="edge"):
    """HW1b's tap loop, 'same' padding, stride 1."""
    img = np.asarray(img, float); kernel = np.asarray(kernel, float)
    kh, kw = kernel.shape
    p = np.pad(img, ((kh // 2, kh // 2), (kw // 2, kw // 2)), mode="edge")
    out = np.zeros_like(img, dtype=float); H, W = img.shape
    for i in range(kh):
        for j in range(kw):
            if kernel[i, j]:
                out += kernel[i, j] * p[i:i + H, j:j + W]
    return out


def gaussian_1d(sigma, truncate=3.0):
    r = int(truncate * sigma); ax = np.arange(-r, r + 1)
    g = np.exp(-(ax ** 2) / (2 * sigma ** 2))
    return g / g.sum()


def gaussian_blur(img, sigma, truncate=3.0):
    """The separable blur from HW1b Task 1.3: a row pass, then a column pass."""
    g = gaussian_1d(sigma, truncate)
    if img.ndim == 3:
        return np.stack([gaussian_blur(img[..., c], sigma, truncate)
                         for c in range(img.shape[2])], axis=-1)
    return convolve2d_fast(convolve2d_fast(img, g[None, :]), g[:, None])


SOBEL_X = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=float)
SOBEL_Y = SOBEL_X.T
print("setup ok - HW1b helpers loaded")

---
## The two slides

| | stain | what is in it |
|---|---|---|
| **A** | **H&E** (haematoxylin and eosin) | chronic alcoholic cirrhosis. Pink cytoplasm, purple nuclei, magenta red blood cells, white sinusoids, a fibrous band running diagonally |
| **B** | **Masson trichrome + Verhoeff** | non-alcoholic fatty liver disease. Red hepatocytes, **green** fibrosis, white round fat vacuoles |

H&E is the default stain in pathology, and the reason it works is chemistry:

- **haematoxylin** is basic and binds DNA, so it stains **nuclei blue-purple**,
- **eosin** is acidic and binds proteins, so it stains **cytoplasm pink**.

That colour split is not decoration. It is the entire signal every task below depends on.

In [ ]:
he = load_image("data/liver_he_cirrhosis.jpg")
tri = load_image("data/liver_trichrome_nafld.jpg")
print("slide A (H&E)      ", he.shape)
print("slide B (trichrome)", tri.shape)
show([he, tri], ["A - H&E, cirrhosis", "B - trichrome, NAFLD"], ncols=2, size=5.2)

# crops: the pure-numpy convolution does not need the full frame
HE = he[300:900, 350:950]
TRI = tri[250:850, 300:900]
show([HE, TRI], ["A, working crop 600x600", "B, working crop 600x600"], ncols=2, size=4.4)

## Task 1 - blur as denoising, and the price 🧀🧀

Stained tissue is speckled: the stain is grainy, the camera adds noise, and individual cells
have texture that is not the structure you are after. A small Gaussian cleans that up.

Push it further and the nuclei merge into each other. Somewhere between those two is the
`sigma` you want, and **nothing in the image tells you where** - only what you intend to
measure next does.

In [ ]:
HE_g = to_gray(HE)
sigmas = [1, 2, 4, 8]
blurs = [gaussian_blur(HE_g, s) for s in sigmas]
show([HE_g] + blurs, ["original"] + [f"sigma = {s}" for s in sigmas], ncols=5, size=2.7,
     suptitle="Denoising, and then over-denoising")

print(f"original local contrast (std) = {HE_g.std():.4f}\n")
print(f"{'sigma':>6}{'detail removed':>18}{'% of original':>16}")
for s, b in zip(sigmas, blurs):
    d = np.std(HE_g - b)
    print(f"{s:>6}{d:>18.4f}{100*d/HE_g.std():>15.1f}%")

In [ ]:
# zoom in on one clump so the merging is visible rather than asserted
z = (slice(180, 300), slice(180, 300))
show([HE[z]] + [gaussian_blur(HE_g, s)[z] for s in sigmas],
     ["original (zoom)"] + [f"sigma = {s}" for s in sigmas], ncols=5, size=2.7,
     suptitle="Same sweep, zoomed: by sigma=8 separate nuclei have become one blob")

## Task 2 - why one edge kernel is never enough 🧀🧀

`Sobel X` fires on **vertical** edges, `Sobel Y` on **horizontal** ones. The fibrous band in
this slide runs diagonally, and cell walls run in every direction at once.

Denoise first - an edge detector applied to raw speckle mostly detects the speckle.

In [ ]:
sm = gaussian_blur(HE_g, 1.5)
gx = convolve2d_fast(sm, SOBEL_X)
gy = convolve2d_fast(sm, SOBEL_Y)
mag = np.sqrt(gx ** 2 + gy ** 2)

show([sm, np.abs(gx), np.abs(gy), mag / mag.max()],
     ["denoised (sigma=1.5)", "|Sobel X|  vertical edges", "|Sobel Y|  horizontal edges",
      "magnitude - both together"], ncols=4, size=3.2)

strong = mag > np.percentile(mag, 99)
ang = np.rad2deg(np.arctan2(np.abs(gy), np.abs(gx)))[strong]
print("orientation of the strongest 1% of edge pixels:")
for lo, hi, lab in [(0, 30, "near-vertical   (Sobel X sees it)"),
                    (30, 60, "diagonal        (needs both)"),
                    (60, 90, "near-horizontal (Sobel Y sees it)")]:
    print(f"  {lab:<34}{100*np.mean((ang >= lo) & (ang < hi)):5.1f}%")
print(f"\nedge energy recovered by a single axis: "
      f"X {100*np.abs(gx)[strong].sum()/mag[strong].sum():.0f}%, "
      f"Y {100*np.abs(gy)[strong].sum()/mag[strong].sum():.0f}%")
print("On a perfect diagonal each axis alone sees 1/sqrt(2) = 71% of the gradient.")

In [ ]:
# what a Sobel-derived edge map looks like at a few denoising levels: the knob matters here too
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
for ax, s in zip(axes, (0.5, 1.5, 3.0, 6.0)):
    b = gaussian_blur(HE_g, s)
    m = np.sqrt(convolve2d_fast(b, SOBEL_X) ** 2 + convolve2d_fast(b, SOBEL_Y) ** 2)
    ax.imshow(m / m.max()); ax.set_title(f"denoise sigma = {s}", fontsize=10); ax.axis("off")
fig.suptitle("Too little denoising detects grain; too much detects nothing",
             fontsize=12, y=1.02)
fig.tight_layout(); plt.show()

## Task 3 - separate the two stains 🧀🧀🧀

The hero task. Nuclei are purple, cytoplasm is pink, and you want a mask of just the nuclei.

**Try RGB first, and watch how it fails** - the failure is more interesting than a flat
refusal to work.

Purple and pink are genuinely close in RGB: both are high in red and high in blue, differing
mostly in how much green they have and in how dark they are. So the obvious RGB rule -
"bluer than green, and dark" - ends up leaning almost entirely on the **darkness** term. It is
a brightness threshold wearing a disguise.

You can tune it to report a believable-looking area. That is exactly the trap. Watch what
happens to the answer when you move that one threshold slightly.

In [ ]:
hsv_he = rgb_to_hsv(HE)
Hh, Ss, Vv = hsv_he[..., 0], hsv_he[..., 1], hsv_he[..., 2]

# what the two populations actually look like in RGB
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.2))
for ax, ch, name in zip(axes, [HE[..., 0], HE[..., 1], HE[..., 2]], "RGB"):
    ax.hist(ch.ravel(), bins=120, color="#0033A0"); ax.set_title(f"{name} channel", fontsize=10)
fig.suptitle("RGB channels: one broad hump each, no clean split anywhere", fontsize=12, y=1.03)
fig.tight_layout(); plt.show()

thrs = (0.55, 0.62, 0.70)
areas = []
for thr in thrs:
    m = (HE[..., 2] > HE[..., 1]) & (to_gray(HE) < thr)
    areas.append(100 * m.mean())
    print(f"RGB rule 'bluer than green AND darker than {thr}': flags {areas[-1]:5.1f}% of the field")
print(f"\nA {max(thrs)-min(thrs):.2f} change in one threshold moves the answer "
      f"{max(areas)/min(areas):.0f}x ({min(areas):.1f}% -> {max(areas):.1f}%).")
show([HE] + [((HE[..., 2] > HE[..., 1]) & (to_gray(HE) < t)) for t in thrs],
     ["crop"] + [f"RGB rule, darkness < {t}" for t in thrs], ncols=4, size=3.2,
     suptitle="The RGB rule is really a brightness threshold wearing a disguise")

### Now in HSV

Look at the **hue** distribution before choosing anything. Everything stained sits in a narrow
arc of magenta, and the two populations are a **shoulder and a peak** inside it - not two
separate islands.

One extra guard is needed. The white sinusoids have near-zero saturation, and hue is an angle
whose value is meaningless when the radius is zero, so their hue is effectively random. That
is the same effect you saw in HW1b, where the H panel went to noise across the pale regions.
Add a saturation floor.

In [ ]:
SAT_FLOOR = 0.18

fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
axes[0].hist(Hh[Ss > SAT_FLOOR].ravel() * 360, bins=120, range=(250, 350), color="#0033A0")
axes[0].axvline(295, color="k", ls="--", lw=1.4)
axes[0].set_title("hue where S > 0.18   (dashed = the valley at ~295 deg)", fontsize=10)
axes[0].set_xlabel("hue (degrees)")
axes[1].hist(Ss.ravel(), bins=120, color="#D90012")
axes[1].axvline(SAT_FLOOR, color="k", ls="--", lw=1.4)
axes[1].set_title("saturation   (dashed = the floor)", fontsize=10)
fig.tight_layout(); plt.show()

pct = np.percentile(Hh[Ss > SAT_FLOOR] * 360, [1, 25, 50, 75, 99])
print("hue percentiles (deg), stained pixels only: " +
      "  ".join(f"p{q}={v:.0f}" for q, v in zip([1, 25, 50, 75, 99], pct)))
print("\nhaematoxylin (nuclei)  = the ~275-295 shoulder")
print("eosin (cytoplasm)      = the ~305-320 peak")
print("the two are ~20 degrees apart - invisible in RGB, obvious on this axis")

In [ ]:
# YOUR CODE HERE - read the three cuts off the histograms above.
# HUE_LO / HUE_HI in turns (degrees / 360); V_MAX is the brightness ceiling.
HUE_LO, HUE_HI, V_MAX = None, None, None
hsv_mask = (Hh > HUE_LO) & (Hh < HUE_HI) & (Ss > SAT_FLOOR) & (Vv < V_MAX)

# build the rule one condition at a time, so each term's job is visible
m1 = (Hh > HUE_LO) & (Hh < HUE_HI)
m2 = m1 & (Ss > SAT_FLOOR)
m3 = m2 & (Vv < V_MAX)
show([HE, m1, m2, m3],
     ["crop",
      f"hue band only: {100*m1.mean():.1f}%",
      f"+ saturation floor: {100*m2.mean():.1f}%",
      f"+ brightness cap: {100*m3.mean():.1f}%"], ncols=4, size=3.2,
     suptitle="Adding one condition at a time")
print(f"the saturation floor removed {100*(m1.mean()-m2.mean()):.1f}% of the field "
      f"(pale pixels whose hue was numerically meaningless)")
print(f"the brightness cap removed a further {100*(m2.mean()-m3.mean()):.1f}%")

## Task 4 - search the thresholds, and notice what is missing 🧀🧀🧀

Where did `295` and `0.78` come from?

From a **search**: try a grid of values, look at what each produces, keep the one that looks
right. That is exactly the hyperparameter search from **[08] Hyperparameter tuning** - grid
search over two knobs.

Except for one thing, and it is the entire point of this task.

In [08] you searched against a **validation set**. Every candidate got a number - accuracy,
AUC, RMSE - and you kept the best number. Here **nobody has labelled a single nucleus**. There
is no score to maximise. So you cannot ask "which is best", only "which is **plausible**", and
plausibility has to be argued from outside the image:

- **internal consistency** - nuclei in one field are roughly the same size, so a setting whose
  components span two orders of magnitude is merging things. Watch the median size, not only
  the count.
- **a rough prior on area** - nuclei are a small minority of a liver field. Single-digit
  percent, not a quarter of it.
- **stability** - if nudging a knob by a few percent moves the answer by half, the answer was
  an artefact of where you stopped searching.

Note what is *not* on that list: the physical size of a nucleus in microns. That would be the
strongest check available, but converting it to pixels needs the **scale bar or the objective
magnification**, and neither image ships with one. Worth noticing how much weaker your
evidence is for the want of one line of metadata.

In [ ]:
MIN_PX = 30


def nuclei_stats(hue_hi, v_max, hue_lo=HUE_LO, sat_floor=SAT_FLOOR, min_px=MIN_PX):
    """Area %, component count and median component size for one threshold setting."""
    # YOUR CODE HERE - build the mask, clean it, label it, and return (area %, kept component count, median component size)
    raise NotImplementedError("build the mask, clean it, label it, and return (area %, kept component count, median component size)")

HUE_GRID = [285, 290, 295, 300, 305]
V_GRID = [0.70, 0.74, 0.78, 0.82]

print("each cell = area% / components / median component px\n")
print("hue<   " + "".join(f"{f'V<{v}':>22}" for v in V_GRID))
for hi in HUE_GRID:
    row = f"{hi:>4}   "
    for v in V_GRID:
        a, n, med = nuclei_stats(hi / 360, v)
        row += f"{f'{a:5.1f}% / {n:3d} / {med:4.0f}':>22}"
    print(row)
print(f"\nchosen: hue < {HUE_HI*360:.0f} deg, V < {V_MAX}"
      f"  ->  {nuclei_stats(HUE_HI, V_MAX)[0]:.1f}% of the field, "
      f"{nuclei_stats(HUE_HI, V_MAX)[1]} components")

In [ ]:
# see the corners of the grid, not just the numbers
picks = [(285, 0.70), (305, 0.70), (285, 0.82), (305, 0.82), (295, 0.78)]
masks, labs = [], []
for hi, v in picks:
    masks.append((Hh > HUE_LO) & (Hh < hi / 360) & (Ss > SAT_FLOOR) & (Vv < v))
    a, n, _ = nuclei_stats(hi / 360, v)
    labs.append(f"hue<{hi}, V<{v}\n{a:.1f}%, n={n}")
show([HE] + masks, ["crop"] + labs, ncols=6, size=2.5,
     suptitle="The corners of the search grid, and the setting we kept (far right)")

### Stability

A threshold you can trust should not be sitting on a cliff. Perturb each knob and see how far
the answer moves.

In [ ]:
base = nuclei_stats(HUE_HI, V_MAX)[1]
print(f"baseline count: {base}\n")
print(f"{'perturbation':<24}{'count':>8}{'change':>10}")
for name, hh, vv in [("hue -5 deg", HUE_HI - 5 / 360, V_MAX),
                     ("hue +5 deg", HUE_HI + 5 / 360, V_MAX),
                     ("V -0.04", HUE_HI, V_MAX - 0.04),
                     ("V +0.04", HUE_HI, V_MAX + 0.04)]:
    c = nuclei_stats(hh, vv)[1]
    print(f"{name:<24}{c:>8}{100*(c-base)/base:>9.0f}%")
print("\nHue is comparatively flat. V is a cliff.")
print("Whatever you report, report it knowing which knob it is standing on.")

## Task 5 - count them, then decide what the count is worth 🧀🧀🧀

Clean the mask with a blur and a re-threshold - a poor man's morphological opening, which
drops isolated speckle - then label connected components. `scipy.ndimage.label` is the one
library call in this notebook.

Then read the number sceptically. Two nuclei that touch become **one** component, so this
undercounts, and it undercounts worst exactly where the tissue is densest. Fixing it needs
watershed or an instance-segmentation model, which is **L19**.

In [ ]:
cleaned = gaussian_blur(hsv_mask.astype(float), 1.5) > 0.5
labels, n_raw = ndimage.label(cleaned)
sizes = ndimage.sum(cleaned, labels, range(1, n_raw + 1))
keep = sizes >= MIN_PX
n_kept = int(keep.sum())
med = np.median(sizes[keep])

for lab, val in [("components before size filter", f"{n_raw}"),
                 (f"after dropping < {MIN_PX}px specks", f"{n_kept}"),
                 ("median component size", f"{med:.0f} px"),
                 ("largest component", f"{sizes.max():.0f} px "
                                       f"({sizes.max()/med:.0f}x the median)")]:
    print(f"{lab:<34}: {val}")

big = np.isin(labels, np.where(keep)[0] + 1)
show([HE, hsv_mask, cleaned, big],
     ["crop", "raw mask", "after blur + re-threshold", f"kept components (n = {n_kept})"],
     ncols=4, size=3.2)

fig, ax = plt.subplots(figsize=(7, 2.8))
ax.hist(sizes[keep], bins=40, color="#0033A0")
ax.axvline(med, color="#D90012", ls="--", lw=1.6, label=f"median {med:.0f} px")
ax.set_xlabel("component size (px)"); ax.legend(); ax.set_title(
    "A single population would be one hump. The long right tail is merged nuclei.", fontsize=10)
fig.tight_layout(); plt.show()
print(f"{100*np.mean(sizes[keep] > 3*med):.0f}% of kept components are more than 3x the "
      f"median size - each of those is almost certainly several nuclei counted as one.")

## Task 6 - a different stain, a different question 🧀🧀🧀

Slide B is a **Masson trichrome**: collagen goes green, hepatocytes stay red, and fat vacuoles
appear white because the lipid washed out during processing.

Three visually distinct populations means three regions in hue and saturation - so an area
estimate is a couple of thresholds away. Pathologists really do grade on these: fibrosis stage
and steatosis percentage.

Treat what follows as *the shape* of that measurement, not a diagnosis. Every number depends
entirely on thresholds picked by eye, and you have just seen how much those move.

In [ ]:
hsv_tri = rgb_to_hsv(TRI)
Ht, St, Vt = hsv_tri[..., 0], hsv_tri[..., 1], hsv_tri[..., 2]

fig, ax = plt.subplots(1, 2, figsize=(12, 3.0))
ax[0].hist(Ht[St > 0.15].ravel() * 360, bins=180, color="#0033A0")
ax[0].set_xlabel("hue (degrees)"); ax[0].set_title("slide B hue, S > 0.15", fontsize=10)
ax[1].hist(St.ravel(), bins=120, color="#D90012")
ax[1].axvline(0.12, color="k", ls="--", lw=1.4)
ax[1].set_title("slide B saturation (dashed = the white cut)", fontsize=10)
fig.tight_layout(); plt.show()
print("Two clear hue modes here, unlike slide A: red hepatocytes and green collagen.")
print("The fat vacuoles are not in this histogram at all - they are unstained, so they")
print("fall out on SATURATION, not hue.")

In [ ]:
# YOUR CODE HERE - two masks on slide B, read off the histograms above.
# fat      = unstained: LOW saturation and HIGH value
# fibrosis = the green hue mode, with a saturation floor
fat = None
fibrosis = None
tissue = ~fat

pct_fat = 100 * fat.mean()
pct_fib = 100 * fibrosis.sum() / tissue.sum()
print(f"white / vacuolar area : {pct_fat:5.1f}% of the field")
print(f"green collagen area   : {pct_fib:5.1f}% of stained tissue")

show([TRI, fat, fibrosis],
     ["crop B", f"unstained, bright ({pct_fat:.1f}%)", f"collagen ({pct_fib:.1f}%)"],
     ncols=3, size=3.8)

# how much does the answer depend on where you cut?
print(f"\n{'S cut':>8}{'V cut':>8}{'fat area':>11}")
for sc in (0.10, 0.12, 0.15):
    for vc in (0.75, 0.80, 0.85):
        print(f"{sc:>8}{vc:>8}{100*((St < sc) & (Vt > vc)).mean():>10.1f}%")
print("\nA reasonable-looking range of cuts moves the headline number by several points.")

## Task 7 - the honesty question 🧀

Take slide A's nuclei rule and run it, unchanged, on slide B. Then slide B's collagen rule on
slide A. The cell below does both.

**Write two or three sentences: why does this happen?**

Things worth naming: the two slides use different stains, so "purple" does not denote the same
tissue in each; they were scanned on different equipment under different lighting, so the white
balance differs; the magnifications differ, so a "30 pixel speck" is a different physical size.
Every constant you tuned - the hue bands, the saturation floor, `MIN_PX` - is attached to one
image, not to biology.

**This is the whole chapter's argument.** You just hand-engineered a pipeline that works, on
real data, measuring something real. It does not survive a change of slide. Everything after
L16 is about networks that learn those constants from data instead of having them typed in.

In [ ]:
cross_A_on_B = (Ht > HUE_LO) & (Ht < HUE_HI) & (St > SAT_FLOOR) & (Vt < V_MAX)
cross_B_on_A = (Hh > 0.30) & (Hh < 0.55) & (Ss > 0.15)

print(f"A's nuclei rule   on A: {100*hsv_mask.mean():5.1f}% of pixels")
print(f"A's nuclei rule   on B: {100*cross_A_on_B.mean():5.1f}% of pixels")
print(f"B's collagen rule on B: {100*fibrosis.mean():5.1f}% of pixels")
print(f"B's collagen rule on A: {100*cross_B_on_A.mean():5.1f}% of pixels")
show([HE, cross_B_on_A, TRI, cross_A_on_B],
     ["A", "B's collagen rule on A", "B", "A's nuclei rule on B"], ncols=4, size=3.0,
     suptitle="Neither rule transfers")

In [ ]:
consts = ["denoise sigma = 1.5",
          f"nuclei hue band = {HUE_LO*360:.0f}-{HUE_HI*360:.0f} deg",
          f"saturation floor = {SAT_FLOOR}",
          f"value ceiling = {V_MAX}",
          "mask clean sigma = 1.5",
          "mask re-threshold = 0.5",
          f"MIN_PX = {MIN_PX}",
          "fat: S < 0.12 and V > 0.80",
          "collagen hue band = 108-198 deg",
          "the two crop rectangles"]
print(f"{len(consts)} hand-picked constants in this pipeline:")
for c in consts:
    print(f"  - {c}")
print("\nNot one came from biology. Every one came from looking at these two images.")
print("A CNN has constants too - millions of them - but it reads them off the data.")

---
## Where this goes

| you did | the chapter does |
|---|---|
| chose a denoising `sigma` by eye | learns the filter that best serves the task |
| hand-picked a hue band per stain | learns colour features from labelled examples |
| counted with connected components, merging touching nuclei | **L19**: instance segmentation |
| watched the rules break on slide 2 | the reason any of this needs learning at all |

**Next:** HW1 Part B trains a small CNN on Fashion-MNIST and visualises its first-layer
kernels. Put them next to the zoo you hand-designed in HW1b, and ask which ones the network
found without being told.